In [1]:
import os, json
import pandas as pd
from tqdm import tqdm

ART_DIR = "../artifacts"
GAPS_PATH  = f"{ART_DIR}/gaps_with_clusters.tsv"
PAIRS_PATH = f"{ART_DIR}/cluster_pairs.tsv"
OUT_PATH   = f"{ART_DIR}/ideas_openai.tsv"

gaps = pd.read_csv(GAPS_PATH, sep="\t")
pairs = pd.read_csv(PAIRS_PATH, sep="\t")

print("gaps:", gaps.shape)
print("pairs:", pairs.shape)
display(gaps.head(2))
display(pairs.head(5))


gaps: (62, 6)
pairs: (30, 5)


,id,gap_type,gap_sentence,paragraph_text,confidence,cluster_id
0,2512.19725,future_work,"In particular, our study highlights the need f...",These findings provide concrete guidance for p...,0.95,1
1,2503.17793,future_work,The future work will focus on further pushing ...,The future work will focus on further pushing ...,0.95,7


,cluster_a,cluster_b,cosine_sim,label_a,label_b
0,2,5,0.609657,"deep, learning, deep learning, dnn, networks, ...","learning, model, future, domain, performance, ..."
1,1,9,0.573678,"research, compression, future, investigate, di...","limited, bias, data, analysis, paper, paper li..."
2,1,5,0.561762,"research, compression, future, investigate, di...","learning, model, future, domain, performance, ..."
3,1,2,0.558066,"research, compression, future, investigate, di...","deep, learning, deep learning, dnn, networks, ..."
4,1,4,0.539070,"research, compression, future, investigate, di...","work, future, future work, patient, datasets, ..."


In [9]:
def pick_evidence(df: pd.DataFrame, cluster_id: int, k: int = 4) -> pd.DataFrame:
    cols = ["id", "gap_type", "confidence", "gap_sentence", "paragraph_text"]
    gg = df[df["cluster_id"] == cluster_id].copy()
    gg = gg.sort_values(["confidence"], ascending=False)
    return gg[cols].head(k)

def build_evidence_payload(df: pd.DataFrame, cluster_id: int, k: int = 4) -> list[dict]:
    ev = pick_evidence(df, cluster_id, k=k)
    out = []
    for _, r in ev.iterrows():
        out.append({
            "paper_id": str(r["id"]),
            "gap_type": str(r["gap_type"]),
            "confidence": float(r["confidence"]),
            "gap_sentence": str(r["gap_sentence"]),
            "paragraph_text": str(r["paragraph_text"]),
        })
    return out

# Choose how many pairs to generate ideas for
TOP_N_PAIRS = min(2, len(pairs))
pairs_top = pairs.head(TOP_N_PAIRS).copy()

# Evidence per pair
EVID_K = 3  # keep small to reduce tokens


In [10]:
IDEA_SCHEMA = {
    "type": "object",
    "properties": {
        "pair": {
            "type": "object",
            "properties": {
                "cluster_a": {"type": "integer"},
                "cluster_b": {"type": "integer"},
            },
            "required": ["cluster_a", "cluster_b"],
            "additionalProperties": False
        },
        "idea": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "research_question": {"type": "string"},
                "method_sketch": {"type": "string"},
                "evaluation_plan": {"type": "string"},
                "expected_contribution": {"type": "string"},
                "assumptions_and_risks": {"type": "string"},
                "evidence_used": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "paper_id": {"type": "string"},
                            "gap_sentence": {"type": "string"}
                        },
                        "required": ["paper_id", "gap_sentence"],
                        "additionalProperties": False
                    }
                },
                "confidence": {"type": "number", "minimum": 0, "maximum": 1}
            },
            "required": [
                "title",
                "research_question",
                "method_sketch",
                "evaluation_plan",
                "expected_contribution",
                "assumptions_and_risks",
                "evidence_used",
                "confidence"
            ],
            "additionalProperties": False
        }
    },
    "required": ["pair", "idea"],
    "additionalProperties": False
}

SYSTEM_IDEA = (
    "You are a research planning assistant. "
    "Use ONLY the provided evidence sentences and paragraphs. "
    "Do NOT invent datasets, results, or claims. "
    "Propose exactly one actionable research idea combining both gap themes."
)


In [11]:
def build_idea_prompt(cluster_a: int, cluster_b: int, label_a: str, label_b: str, ev_a: list[dict], ev_b: list[dict]) -> str:
    # Minimal, explicit, constrained
    payload = {
        "cluster_a": cluster_a,
        "cluster_b": cluster_b,
        "theme_a_label": label_a,
        "theme_b_label": label_b,
        "evidence_a": ev_a,
        "evidence_b": ev_b
    }
    return (
        "Return JSON matching schema.\n"
        "Constraints:\n"
        "- Use ONLY evidence_a/evidence_b content.\n"
        "- No fabricated citations, datasets, numbers, or results.\n"
        "- Method must be testable and include evaluation plan.\n"
        "- evidence_used must be a subset of provided evidence sentences.\n"
        "INPUT:\n"
        + json.dumps(payload, ensure_ascii=False)
    )


In [12]:
!pip -q install python-dotenv openai

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # loads .env from current directory
assert os.getenv("OPENAI_API_KEY") is not None, "OPENAI_API_KEY not found in .env"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI key loaded ✅")


OpenAI key loaded ✅


In [13]:
MODEL_IDEA = "gpt-4.1-mini"  # keep consistent with your extraction

idea_rows = []

for _, pr in tqdm(pairs_top.iterrows(), total=len(pairs_top)):
    ca = int(pr["cluster_a"])
    cb = int(pr["cluster_b"])

    label_a = str(pr.get("label_a", ""))
    label_b = str(pr.get("label_b", ""))

    ev_a = build_evidence_payload(gaps, ca, k=EVID_K)
    ev_b = build_evidence_payload(gaps, cb, k=EVID_K)

    # Skip if evidence missing
    if len(ev_a) == 0 or len(ev_b) == 0:
        continue

    resp = client.responses.create(
        model=MODEL_IDEA,
        input=[
            {"role": "system", "content": SYSTEM_IDEA},
            {"role": "user", "content": build_idea_prompt(ca, cb, label_a, label_b, ev_a, ev_b)},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "idea_synthesis",
                "schema": IDEA_SCHEMA,
                "strict": True,
            }
        },
    )

    data = json.loads(resp.output_text)
    idea = data["idea"]

    idea_rows.append({
        "cluster_a": ca,
        "cluster_b": cb,
        "cosine_sim": float(pr.get("cosine_sim", None)) if "cosine_sim" in pr else None,
        "label_a": label_a,
        "label_b": label_b,
        "title": idea["title"],
        "research_question": idea["research_question"],
        "method_sketch": idea["method_sketch"],
        "evaluation_plan": idea["evaluation_plan"],
        "expected_contribution": idea["expected_contribution"],
        "assumptions_and_risks": idea["assumptions_and_risks"],
        "idea_confidence": idea["confidence"],
        # store evidence used compactly
        "evidence_used_json": json.dumps(idea["evidence_used"], ensure_ascii=False),
    })

ideas_df = pd.DataFrame(idea_rows)
print("ideas:", ideas_df.shape)
display(ideas_df.head(10))


100%|██████████| 2/2 [00:19<00:00,  9.57s/it]

ideas: (2, 13)


,cluster_a,cluster_b,cosine_sim,label_a,label_b,title,research_question,method_sketch,evaluation_plan,expected_contribution,assumptions_and_risks,idea_confidence,evidence_used_json
0,2,5,0.609657,"deep, learning, deep learning, dnn, networks, ...","learning, model, future, domain, performance, ...",Scalable Model Merging for Large Deep Learning...,"How can we develop an efficient, scalable, and...",We propose to implement a scalable model mergi...,Evaluation will consist of multiple experiment...,"This research will provide a novel, scalable, ...",Assumes availability and stability of distribu...,0.92,"[{""paper_id"": ""2102.03018"", ""gap_sentence"": ""W..."
1,1,9,0.573678,"research, compression, future, investigate, di...","limited, bias, data, analysis, paper, paper li...",Exploring Hybrid Model Feature Combinations fo...,How can different input shaping techniques com...,We propose to investigate various combinations...,Evaluate the approach by measuring classificat...,This study will deliver a deeper understanding...,Assumes access to annotated datasets with expe...,0.90,"[{""paper_id"": ""2401.08233"", ""gap_sentence"": ""F..."


In [14]:
ideas_df.to_csv(OUT_PATH, sep="\t", index=False)
print("Saved:", OUT_PATH)


Saved: ../artifacts/ideas_openai.tsv


In [15]:
def explode_evidence_used(evidence_used_json: str) -> pd.DataFrame:
    lst = json.loads(evidence_used_json)
    return pd.DataFrame(lst)

if len(ideas_df) > 0:
    row = ideas_df.iloc[0]
    print("TITLE:", row["title"])
    print("RQ:", row["research_question"])
    print("\nMETHOD:\n", row["method_sketch"])
    print("\nEVAL:\n", row["evaluation_plan"])
    print("\nCONTRIB:\n", row["expected_contribution"])
    print("\nRISKS:\n", row["assumptions_and_risks"])
    print("\nEVIDENCE USED:")
    display(explode_evidence_used(row["evidence_used_json"]))


TITLE: Scalable Model Merging for Large Deep Learning Models with Robustness Evaluation
RQ: How can we develop an efficient, scalable, and mathematically principled model merging algorithm for large deep learning models, while incorporating robustness and trustworthiness evaluation tools to ensure deployed model reliability?

METHOD:
 We propose to implement a scalable model merging algorithm designed for large-scale deep learning architectures such as ResNet, leveraging distributed learning frameworks and advanced optimizers like LAMB to handle large datasets (e.g., ImageNet). The method will integrate principles from existing statistical methods like Fisher information and gradient matching to improve merging efficacy. Simultaneously, we will incorporate robustness evaluation mechanisms inspired by tools such as NeuralSentinel, extending them to assess model trustworthiness during and after merging. Custom gradient functions and tailored training schemes (e.g., QFX training) will be 

,paper_id,gap_sentence
0,2102.03018,We plan to extend this task by performing simi...
1,2504.16732,"Notwithstanding these contributions, the devel..."
2,2402.07506,"In the future, the NeuralSentinel tool will be..."
3,2401.17544,Users can also define customized gradient func...
